# MCNN-LST on Fashion-MNIST

End-to-end study notebook for the **MCNN-LST** model. The notebook is organised into eight self-contained sections — you can run them independently once sections 1–4 have been executed.

1. Imports & device setup
2. Dataset (Fashion-MNIST)
3. Model: `MultiConv4_LST`
4. Training / evaluation helpers
5. Load pretrained model and evaluate on the test set
6. Train a single model with custom hyperparameters
7. Hyperparameter search with Optuna
8. 10-run statistics with the best configuration


## 1. Imports & device setup

In [2]:
import os
import shutil

import numpy as np
import torch
import torch.nn as nn
from torch import optim
from torch.optim import lr_scheduler
from torch.nn.functional import max_pool2d, relu
import torchvision
from torchvision import transforms
from torch.utils.tensorboard import SummaryWriter

from l2dst_lib.lst_nn import L2DST, multichan_to_2D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch {torch.__version__}  |  device: {device}")


[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


PyTorch 2.1.2+cpu  |  device: cpu


## 2. Dataset (Fashion-MNIST)

The dataset is downloaded automatically the first time the cell runs. The training set is split into 59 000 train / 1 000 validation samples; the held-out test set is used only for final evaluation.

In [3]:
Dataset_name = 'Fashion'
DATA_DIR = '../Datasets'

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])

trainset_full = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=True, download=True, transform=transform)
testset = torchvision.datasets.FashionMNIST(
    root=DATA_DIR, train=False, download=True, transform=transform)

val_size, train_size = 1000, 59000
val_dataset, train_dataset = torch.utils.data.random_split(
    trainset_full, [val_size, train_size],
    generator=torch.Generator().manual_seed(27))

val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=1000, shuffle=False)
test_loader = torch.utils.data.DataLoader(testset, batch_size=1000, shuffle=False)
print(f"train={len(trainset_full)}  val={len(val_dataset)}  test={len(testset)}")


train=60000  val=1000  test=10000


## 3. Model: `MultiConv4_LST`

Four parallel `Conv2d` columns (kernels 2, 3, 4, 5) feed into per-column `LST` blocks. Their outputs are concatenated, batch-normalised, regularised with dropout, and passed to a linear classifier.

**Constraint:** `c1_kernels` must be a perfect square — the channel dimension is reshaped into a 2-D grid by `multichan_to_2D` before LST.

In [4]:
class MultiConv4_LST(nn.Module):
    def __init__(self, c1_kernels=16, lst_out=4, num_classes=10,
                 p_drop_lst=0.1, p_drop_fc=0.1, device='cpu'):
        super().__init__()
        self.conv1 = nn.Conv2d(1, c1_kernels, 2, padding='same')
        self.conv2 = nn.Conv2d(1, c1_kernels, 3)
        self.conv3 = nn.Conv2d(1, c1_kernels, 4, padding='same')
        self.conv4 = nn.Conv2d(1, c1_kernels, 5)

        self.BN1_1 = nn.BatchNorm2d(1)
        self.BN1_2 = nn.BatchNorm2d(1)
        self.BN1_3 = nn.BatchNorm2d(1)
        self.BN1_4 = nn.BatchNorm2d(1)
        self.BN2 = nn.BatchNorm1d(4 * lst_out * lst_out)

        self.lst_out = lst_out
        s = int(np.sqrt(c1_kernels))
        assert s * s == c1_kernels, "c1_kernels must be a perfect square"

        self.lst1 = L2DST([14 * s, 14 * s], [lst_out, lst_out], device=device, p_drop=p_drop_lst)
        self.lst2 = L2DST([13 * s, 13 * s], [lst_out, lst_out], device=device, p_drop=p_drop_lst)
        self.lst3 = L2DST([14 * s, 14 * s], [lst_out, lst_out], device=device, p_drop=p_drop_lst)
        self.lst4 = L2DST([12 * s, 12 * s], [lst_out, lst_out], device=device, p_drop=p_drop_lst)

        self.fc1 = nn.Linear(4 * lst_out * lst_out, num_classes)
        self.drop1 = nn.Dropout(p=p_drop_fc)
        nn.init.xavier_normal_(self.fc1.weight)

    def forward(self, x):
        x1 = max_pool2d(relu(self.conv1(x)), (2, 2))
        x2 = max_pool2d(relu(self.conv2(x)), (2, 2))
        x3 = max_pool2d(relu(self.conv3(x)), (2, 2))
        x4 = max_pool2d(relu(self.conv4(x)), (2, 2))

        x1 = multichan_to_2D(x1)
        x2 = multichan_to_2D(x2)
        x3 = multichan_to_2D(x3)
        x4 = multichan_to_2D(x4)

        x1 = self.BN1_1(x1.unsqueeze(1)).squeeze(1)
        x2 = self.BN1_2(x2.unsqueeze(1)).squeeze(1)
        x3 = self.BN1_3(x3.unsqueeze(1)).squeeze(1)
        x4 = self.BN1_4(x4.unsqueeze(1)).squeeze(1)

        x1 = self.lst1(x1)
        x2 = self.lst2(x2)
        x3 = self.lst3(x3)
        x4 = self.lst4(x4)

        combined = torch.cat([
            torch.flatten(x1, 1),
            torch.flatten(x2, 1),
            torch.flatten(x3, 1),
            torch.flatten(x4, 1),
        ], dim=1)
        combined = self.BN2(combined)
        return self.fc1(self.drop1(combined))


# Quick sanity check
_m = MultiConv4_LST(c1_kernels=36, lst_out=28, device=device).to(device)
_n_params = sum(p.numel() for p in _m.parameters() if p.requires_grad)
print(f"MultiConv4_LST(c1_kernels=36, lst_out=28) parameters = {_n_params:,}")
del _m


MultiConv4_LST(c1_kernels=36, lst_out=28) parameters = 57,770


## 4. Training & evaluation helpers

In [5]:
def acc_estimate(model, data_loader):
    """Compute classification accuracy of `model` on `data_loader`."""
    model.to(device)
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y_true in data_loader:
            X = X.to(device)
            y_true = y_true.to(device)
            outputs = model(X)
            _, y_pred = torch.max(outputs, dim=1)
            total += y_true.shape[0]
            correct += int((y_pred == y_true).sum())
    return correct / total


def remove_directory(dir_path):
    if os.path.exists(dir_path):
        shutil.rmtree(dir_path)


def train_test_loop(model, model_name, Dataset_name, trainloader, val_loader,
                    test_loader=None, N_epoch=200, weight_decay=1e-6, lr=2e-3,
                    T0=100, eta_min=1e-6, save_best=True):
    """Train `model` and track best accuracy on `val_loader`.

    The checkpoint with the highest validation accuracy is written to
    `model_backup/{Dataset_name}/{model_name}/{model_name}_epoch_{N_epoch}.pth`.
    Returns (best_val_acc, test_acc) where `test_acc` is computed on
    `test_loader` if provided, otherwise equals `best_val_acc`.
    """
    model_path = os.path.join('model_backup', Dataset_name, model_name)
    os.makedirs(model_path, exist_ok=True)
    target_path = os.path.join(model_path, f"{model_name}_epoch_{N_epoch}.pth")

    model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr,
                            weight_decay=weight_decay, amsgrad=True)
    scheduler = lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, T_0=T0, T_mult=1, eta_min=eta_min)

    best_acc = 0.0
    for epoch in range(N_epoch):
        # -- validation --
        model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                labels = labels.to(device)
                output = model(images)
                val_loss += criterion(output, labels).item()
                _, y_pred = torch.max(output, dim=1)
                total += labels.shape[0]
                correct += int((y_pred == labels).sum())
        val_loss /= len(val_loader)
        val_acc = correct / total
        writer.add_scalar('Accuracy (val)', val_acc, epoch)
        writer.add_scalar('Val loss', val_loss, epoch)

        if save_best and val_acc > best_acc:
            torch.save(model.state_dict(), target_path)
            best_acc = val_acc

        # -- training --
        model.train()
        running_loss = 0.0
        for images, labels in trainloader:
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()
        writer.add_scalar('Train loss', running_loss / len(trainloader), epoch)

    if save_best and os.path.exists(target_path):
        model.load_state_dict(torch.load(target_path, map_location=device))
    test_acc = acc_estimate(model, test_loader) if test_loader is not None else best_acc
    print(f"Best val acc = {best_acc:.4f},  test acc = {test_acc:.4f}")
    return best_acc, test_acc


## 5. Load pretrained model and evaluate on the test set

Loads the checkpoint shipped under `model_backup/Fashion/` and reports its accuracy on the Fashion-MNIST test set.

In [6]:
# Architecture hyperparameters of the released checkpoint
LST_OUT      = 28
KERNEL_NUM   = 36
P_DROP_FC    = 0.169
P_DROP_LST   = 0.198

CHECKPOINT_NAME = 'MCNN_LST_28_ker36'
CHECKPOINT_PATH = os.path.join(
    'pretrain', f'{CHECKPOINT_NAME}.pth')

print(f"Loading {CHECKPOINT_PATH}")
model = MultiConv4_LST(
    c1_kernels=KERNEL_NUM, lst_out=LST_OUT,
    p_drop_fc=P_DROP_FC, p_drop_lst=P_DROP_LST, device=device,
).to(device)
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
model.eval()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
acc_test = acc_estimate(model, test_loader)
acc_val  = acc_estimate(model, val_loader)
print(f"Parameters    = {n_params:,}")
print(f"Test accuracy = {acc_test:.4f}")
print(f"Val accuracy  = {acc_val:.4f}")


Loading pretrain\MCNN_LST_28_ker36.pth


c:\Users\m.vashkevich\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\modules\conv.py:456: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at ..\aten\src\ATen\native\Convolution.cpp:1009.)
  return F.conv2d(input, weight, bias, self.stride,


Parameters    = 57,770
Test accuracy = 0.9358
Val accuracy  = 0.9950


## 6. Train a single model with custom hyperparameters

Adjust the constants below and run the cell to train one model from scratch. The best checkpoint (by validation accuracy) is written to `model_backup/Fashion/<MODEL_NAME>/`.

In [ ]:
# --- user-tunable hyperparameters -----------------------------------------
LR           = 4.59e-03
WD           = 1.92e-06
P_DROP_FC    = 0.169
P_DROP_LST   = 0.198
T_0_LOOPS    = 2
LST_OUT      = 28
KERNEL_NUM   = 36
BATCH_SIZE   = 2048
N_EPOCHS     = 200
ETA_MIN      = LR * 0.01
RANDOM_SEED  = 27
# --------------------------------------------------------------------------

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

model_name = f'MultiConv4_LST_{LST_OUT}_ker{KERNEL_NUM}_custom'
model = MultiConv4_LST(
    c1_kernels=KERNEL_NUM, lst_out=LST_OUT,
    p_drop_fc=P_DROP_FC, p_drop_lst=P_DROP_LST, device=device,
).to(device)

T_0 = N_EPOCHS // T_0_LOOPS
TENSORBOARD_LOGS_DIR = (
    f"runs/{model_name}/lr{LR:1.2e}-wd{WD:1.2e}-T0_{T_0}")
writer = SummaryWriter(TENSORBOARD_LOGS_DIR)

trainloader = torch.utils.data.DataLoader(
    trainset_full, batch_size=BATCH_SIZE, shuffle=True)

best_val, acc_test = train_test_loop(
    model, model_name, Dataset_name,
    trainloader, val_loader, test_loader=test_loader,
    N_epoch=N_EPOCHS, weight_decay=WD, lr=LR,
    T0=T_0, eta_min=ETA_MIN)
writer.close()
print(f"Saved best checkpoint under model_backup/Fashion/{model_name}/")


## 7. Hyperparameter search with Optuna

The search space matches the one reported in the paper. Trials are persisted in a local SQLite database, so the optimisation can be paused and resumed safely.

In [ ]:
import optuna

SQLITE_STORAGE_URL = "sqlite:///db_fashion.sqlite3"
STUDY_NAME         = "j-iet-MCNN_LST-FashionMNIST"
N_TRIALS           = 100
N_EPOCHS_OPTUNA    = 200


def objective(trial: optuna.Trial) -> float:
    lr         = trial.suggest_float("lr", 1e-4, 3e-2, log=True)
    wd         = trial.suggest_float("wd", 1e-8, 1e-4, log=True)
    p_drop_fc  = trial.suggest_float("p_drop_fc",  0.01, 0.55)
    p_drop_lst = trial.suggest_float("p_drop_lst", 0.01, 0.55)
    t_0_loop   = trial.suggest_categorical("t_0_loop", [20, 10, 4, 2])

    lst_out    = 28
    kernel_num = 36
    batch_size = 2048
    eta_min    = lr * 0.01

    torch.manual_seed(27)
    np.random.seed(27)

    model_name = f'MultiConv4_LST_{lst_out}_ker{kernel_num}_t{trial.number}'
    model = MultiConv4_LST(
        c1_kernels=kernel_num, lst_out=lst_out,
        p_drop_fc=p_drop_fc, p_drop_lst=p_drop_lst, device=device,
    ).to(device)

    T_0 = N_EPOCHS_OPTUNA // t_0_loop

    global TENSORBOARD_LOGS_DIR, writer
    TENSORBOARD_LOGS_DIR = (
        f"runs/optuna/{model_name}-lr{lr:1.2e}-wd{wd:1.2e}-T0_{T_0}")
    writer = SummaryWriter(TENSORBOARD_LOGS_DIR)

    trainloader = torch.utils.data.DataLoader(
        trainset_full, batch_size=batch_size, shuffle=True)
    best_val, _ = train_test_loop(
        model, model_name, Dataset_name,
        trainloader, val_loader, test_loader=None,
        N_epoch=N_EPOCHS_OPTUNA, weight_decay=wd, lr=lr,
        T0=T_0, eta_min=eta_min)
    writer.close()
    return best_val


study = optuna.create_study(
    storage=SQLITE_STORAGE_URL,
    study_name=STUDY_NAME,
    direction='maximize',
    load_if_exists=True,
)


In [ ]:
# Launch / resume the search (set N_TRIALS to 0 above to just inspect the study)
study.optimize(objective, n_trials=N_TRIALS)
print("Best params:", study.best_params)
print(f"Best value : {study.best_value:.4f}")


## 8. 10-run statistics with the best configuration

Trains the best configuration ten times with different RNG seeds and reports mean ± standard deviation of test accuracy.

In [ ]:
N_RUNS       = 10
N_EPOCHS_RUN = 200
BATCH_SIZE   = 2048

# Either pull the best params from the Optuna study above ...
try:
    best_params = study.best_params
    LR_BEST         = best_params["lr"]
    WD_BEST         = best_params["wd"]
    P_DROP_FC_BEST  = best_params["p_drop_fc"]
    P_DROP_LST_BEST = best_params["p_drop_lst"]
    T_0_LOOP_BEST   = best_params["t_0_loop"]
    print("Loaded best params from Optuna study.")
except (NameError, ValueError):
    # ... or fall back to the values reported in the paper.
    LR_BEST, WD_BEST     = 4.59e-03, 1.92e-06
    P_DROP_FC_BEST       = 0.169
    P_DROP_LST_BEST      = 0.198
    T_0_LOOP_BEST        = 2
    print("Using paper-reported best params.")

LST_OUT      = 28
KERNEL_NUM   = 36
ETA_MIN      = LR_BEST * 0.01
T_0          = N_EPOCHS_RUN // T_0_LOOP_BEST

HEADER_FMT = "| {:36} | {:>7} | {:>6} |"
SEP_LINE   = "|" + "-" * 38 + "|" + "-" * 9 + "|" + "-" * 8 + "|"

results_path = f'results_MultiConv4_LST-{LST_OUT}-k{KERNEL_NUM}.txt'
acc_list = []
with open(results_path, 'w') as f:
    print(HEADER_FMT.format("model_name", "params", "ACC"), file=f)
    print(SEP_LINE, file=f)
    for run in range(N_RUNS):
        torch.manual_seed(27 + run)
        np.random.seed(27 + run)

        model_name = (
            f'MultiConv4_LST-{LST_OUT}-ker{KERNEL_NUM}'
            f'-bs{BATCH_SIZE}-T0_{T_0}_run{run}')
        model = MultiConv4_LST(
            c1_kernels=KERNEL_NUM, lst_out=LST_OUT,
            p_drop_fc=P_DROP_FC_BEST, p_drop_lst=P_DROP_LST_BEST,
            device=device,
        ).to(device)

        if run == 0:
            n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

        TENSORBOARD_LOGS_DIR = f"runs/stat/fashion/{model_name}"
        remove_directory(TENSORBOARD_LOGS_DIR)
        writer = SummaryWriter(TENSORBOARD_LOGS_DIR)

        trainloader = torch.utils.data.DataLoader(
            trainset_full, batch_size=BATCH_SIZE, shuffle=True)
        _, acc = train_test_loop(
            model, model_name, Dataset_name,
            trainloader, val_loader, test_loader=test_loader,
            N_epoch=N_EPOCHS_RUN, weight_decay=WD_BEST, lr=LR_BEST,
            T0=T_0, eta_min=ETA_MIN)
        writer.close()
        acc_list.append(acc)
        line = f"| {model_name:36} | {n_params:7,} | {acc:6.4f} |"
        print(line, file=f)
        print(line)

    acc_mean = float(np.mean(acc_list))
    acc_std  = float(np.std(acc_list))
    summary = (f"| FINAL  mean +/- std (n={N_RUNS:2d})              "
               f"| {n_params:7,} | {acc_mean:6.4f} +/- {acc_std:6.4f} |")
    print(summary, file=f)
    print(summary)
print(f"Results written to {results_path}")
